[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session0/BMEN35_Ex0_02_NumPy.ipynb)

# NumPy and numerical computing

In this notebook we will look at NumPy, one of the libraries you will use often in scientific Python. NumPy gives us arrays for storing numerical data and functions for working with those arrays.

If you use Google Colab, NumPy is already installed. If you run this notebook locally, you will need to install it yourself. The cells below are written out slowly so we can see how an array is made and what its shape means.

## Import NumPy

It is common to import NumPy with the short name `np`. You will see this in many examples and in the course notebooks.

In [ ]:
import numpy as np

## Make some arrays

A NumPy array can be made from a Python list. NumPy arrays usually contain values of the same type. This makes them useful for numerical calculations.

In [ ]:
a = np.array([1, 2, 3])
print(type(a))
print(a)

We can also make arrays filled with zeros or ones. This is useful when we want to initialize a matrix before filling it with values.

In [ ]:
zeros = np.zeros(5)
ones = np.ones(5)
print('Zeros:', zeros)
print('Ones:', ones)

A two-dimensional array has rows and columns. The shape `(4,  3)` means four rows and three columns. The shape is given as a tuple of integers.

In [ ]:
matrix = np.zeros((4, 3))
print(matrix)
print('Shape:', matrix.shape)

## NumPy vocabulary: shape, dimensions and size

Each direction in an array is an **axis**. The `ndim` attribute tells us how many axes there are. The `shape` tells us the size along each axis, and `size` tells us the total number of values.

In [ ]:
values = np.ones((5, 3))
print(values)
print('Number of dimensions:', values.ndim)
print('Shape:', values.shape)
print('Total number of values:', values.size)

The first shape value is the number of rows, and the second is the number of columns. Let's select a row and then one value from the matrix. As in Python lists, indexing starts at zero.

In [ ]:
measurements = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
])

print('Whole array:')
print(measurements)
print('Second row:', measurements[1, :])
print('Value in row 3, column 2:', measurements[2, 1])

## Create a sequence of values

`np.arange` is similar to Python's `range`. It starts at the first number and stops before the final number. Here we step by two each time.

In [ ]:
sequence = np.arange(1, 10, 2)
print(sequence)

## Calculate on every value

One useful thing about NumPy is that we can do arithmetic on a whole array. The calculation is applied to each value. We do not have to write a loop for this simple operation.

In [ ]:
measurements = np.array([72, 75, 71, 78])
print('Measurements:', measurements)
print('Measurements plus one:', measurements + 1)
print('Squared values:', np.square(measurements))

NumPy also has methods for common summaries. Let's calculate the mean, sum and standard deviation. For a 2D array, we need to decide which axis we are summarizing.

In [ ]:
print('Mean of all values:', measurements.mean())
print('Sum of all values:', measurements.sum())

measurements_2d = np.array([[72, 98], [75, 96], [71, 101], [78, 94]])
print('Shape:', measurements_2d.shape)
print('Mean for each column:', measurements_2d.mean(axis=0))
print('Mean for each row:', measurements_2d.mean(axis=1))

`axis=0` moves down the rows and gives one result for each column. `axis=1` moves across the columns and gives one result for each row. This can take a little practice, so check the shape of the result as well as the numbers.

## Element-wise multiplication and the dot product

Let's start with two short vectors:

$$
\mathbf{a}=\begin{bmatrix}2 \\ 3\end{bmatrix},
\qquad
\mathbf{b}=\begin{bmatrix}4 \\ 1\end{bmatrix}.
$$

There are two different operations we might want. Element-wise multiplication multiplies matching positions. The dot product multiplies matching positions **and then adds** the results. NumPy uses `*` for the first operation and `np.dot` (or `@`) for the second one.

In [ ]:
a = np.array([2, 3])
b = np.array([4, 1])

# * multiplies the first pair and the second pair separately.
elementwise_result = a * b
print('Element-wise result:', elementwise_result)

# The dot product adds those two products: (2 * 4) + (3 * 1).
dot_result = np.dot(a, b)
print('Dot product:', dot_result)

The first result is still a vector with two values: $[2\times4,\;3\times1]=[8,3]$. The dot product is one number: $2\times4+3\times1=11$.

Both operations are useful, but they are not interchangeable. Let's see why the dot product appears in a simple linear model.

## A linear model written with matrices

A linear model combines the features in one observation with model weights. With several observations, we can write all the calculations together:

$$
\hat{\mathbf{y}} = \mathbf{X}\boldsymbol{\theta}.
$$

Here is a small example. Each row of $\mathbf{X}$ is one observation. Each column is one feature. The first column is all ones so the first weight acts as an intercept:

$$
\mathbf{X}=\begin{bmatrix}
1 & 2.4 & 3.2\\
1 & 3.4 & 0.9\\
1 & 2.3 & 4.1
\end{bmatrix},
\qquad
\boldsymbol{\theta}=\begin{bmatrix}7.4\\3.2\\4.1\end{bmatrix}.
$$

The matrix has shape $(3,3)$, and there are three weights. Each row produces one prediction, so we expect three predicted values.

In [ ]:
# Each row is one observation; each column is one feature.
X = np.array([
    [1, 2.4, 3.2],
    [1, 3.4, 0.9],
    [1, 2.3, 4.1]
])

# One model weight for each column in X.
theta = np.array([7.4, 3.2, 4.1])

print('X:')
print(X)
print('Shape of X:', X.shape)
print('theta:', theta)
print('Shape of theta:', theta.shape)

## What happens if we use `*` here?

The expression `X * theta` looks plausible, but it does **not** calculate the three predictions. NumPy repeats the three weights across each row and multiplies each entry separately. The result still has shape $(3,3)$.

This is a good example of why we should check both the values and the shape of an array. A calculation can run without an error and still be the wrong calculation for our purpose.

In [ ]:
# This is element-wise multiplication, not matrix multiplication.
wrong_shape_for_predictions = X * theta
print(wrong_shape_for_predictions)
print('Shape after X * theta:', wrong_shape_for_predictions.shape)

# We have three observations, so we expected three predictions, not a 3-by-3 matrix.

## Calculate one prediction for each row

For the linear model we want the dot product of each row with the weight vector. The matrix multiplication rule says the inside dimensions must match:

$$
(3\times3)(3\times1) = (3\times1).
$$

In code, `np.dot(X, theta)` and `X @ theta` both calculate the three predictions when `theta` is a one-dimensional array.

In [ ]:
# np.dot multiplies each row of X by theta and sums the three products.
y_hat = np.dot(X, theta)
print('Predictions from np.dot:', y_hat)
print('Shape of predictions:', y_hat.shape)

# The @ symbol is another way to write matrix multiplication.
y_hat_with_at = X @ theta
print('Predictions from @:', y_hat_with_at)

The weights and input values are just teaching numbers. For the first row the calculation is:

$$
7.4\times1 + 3.2\times2.4 + 4.1\times3.2 = 28.68.
$$

The first item in `y_hat` should match that calculation. We can calculate it by hand and compare it with NumPy.

In [ ]:
first_prediction_by_hand = 7.4 * 1 + 3.2 * 2.4 + 4.1 * 3.2
print('First prediction by hand:', first_prediction_by_hand)
print('First prediction from NumPy:', y_hat[0])

## One-dimensional, row and column vectors

The shape of `theta` is `(3,)`. That means it is a one-dimensional array with three values. It is **not** explicitly a row vector or a column vector.

If we reshape it to `(1, 3)`, it becomes a row vector. In the product $X\theta$, the number of columns in $X$ must match the number of rows in $\theta$. A `(3, 3)` matrix cannot multiply a `(1, 3)` row vector in that order.

In [ ]:
# Make an explicit row vector with one row and three columns.
theta_row = theta.reshape(1, 3)
print(theta_row)
print('Shape of the row vector:', theta_row.shape)

# Show the dimension error without stopping the notebook.
try:
    X @ theta_row
except ValueError as error:
    print('X @ theta_row does not work:')
    print(error)

To multiply $X$ by the weights as an explicit matrix, we need a **column vector** with shape `(3, 1)`. We can get it by transposing the row vector. The result then has three rows and one column: one prediction for each observation.

In [ ]:
# Transpose changes the shape from (1, 3) to (3, 1).
theta_column = theta_row.T
print(theta_column)
print('Shape of the column vector:', theta_column.shape)

# The inner dimensions now match: (3, 3) @ (3, 1) gives (3, 1).
predictions_column = X @ theta_column
print('Predictions as a column:')
print(predictions_column)
print('Shape of the result:', predictions_column.shape)

## Matrix multiplication: check the shapes first

The general rule is:

$$
(m\times n)(n\times p) = (m\times p).
$$

The two inside numbers must match. The outside numbers tell us the shape of the answer. Let's use a different example with two matrices so we can see the rule directly.

In [ ]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

B = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
])

print('Shape of A:', A.shape)  # 2 rows, 3 columns
print('Shape of B:', B.shape)  # 3 rows, 2 columns

C = A @ B
print('A @ B:')
print(C)
print('Shape of C:', C.shape)  # 2 rows, 2 columns

Notice that `A * B` would try to multiply matching entries, but the two arrays do not even have the same shape. `A @ B` works because the three columns of `A` match the three rows of `B`. Check the dimensions before trying matrix multiplication; the shape tells us whether the operation makes sense.

## Make a small signal

NumPy arrays are also handy for signals. Here we make a time axis and add two sine waves together. This is only a simple generated signal for learning array operations; it is not an ECG simulator.

In [ ]:
time = np.linspace(0, 4, 800)
slow_wave = np.sin(2 * np.pi * time)
small_wave = 0.2 * np.sin(2 * np.pi * 3 * time)
signal = slow_wave + small_wave

print('Number of time samples:', len(time))
print('Signal shape:', signal.shape)

## Pause and check

1. What does a shape of `(4, 3)` mean?
2. How is `a * b` different from `np.dot(a, b)`?
3. What does `axis=0` mean when we calculate a mean from a table?
4. Why should we look at the shape of an array before using it in a model?

Next we will use Matplotlib to plot arrays like the signal we just made.